In [2]:
from pathlib import Path
import pandas as pd 
import numpy as np
from torchinfo import summary

from src.models import CUSTOM_MODELS
from src.models.scratch_cnn import ConvNet
from src.models.finetuning import inject_linear_mlp_probing,inject_lora_transformer


/Users/co/Code/DataChallenge2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
N_BINS = 30
ALPHA_SMOOTH = 50

def compute_gender_weights(df, n_bins=N_BINS, alpha=ALPHA_SMOOTH):
    """
    Calcule des poids par bin × genre pour équilibrer les distributions H/F.
    Retourne : bins (frontières), w_f (poids femmes par bin), w_m (poids hommes).
    """
    gt     = df["FaceOcclusion"].values
    gender = df["gender"].values

    bins    = np.linspace(0, 1, n_bins + 1)
    bin_idx = np.clip(np.digitize(gt, bins, right=False) - 1, 0, n_bins - 1)

    n_f = np.zeros(n_bins)
    n_m = np.zeros(n_bins)
    for b in range(n_bins):
        mask = bin_idx == b
        n_f[b] = np.sum((gender == 0.0) & mask)
        n_m[b] = np.sum((gender == 1.0) & mask)

    n_total = n_f + n_m
    w_f = (n_total + 2 * alpha) / (2 * (n_f + alpha))
    w_m = (n_total + 2 * alpha) / (2 * (n_m + alpha))

    return bins, w_f, w_m

In [4]:
df_train = pd.read_csv("../data/occlusion_datasets/train.csv", delimiter=',')
df_test = pd.read_csv("../data/occlusion_datasets/test_students.csv", delimiter=',')

image_dir = "crops/Crop_224_5fp_100K"

In [5]:
bins, w_f, w_m = compute_gender_weights(df_train, n_bins=N_BINS, alpha=ALPHA_SMOOTH)

In [6]:
df_train.head()

,filename,FaceOcclusion,gender
0,database3/database3/m.01w0zk1/66-FaceId-0_alig...,0.024005,1.0
1,database3/database3/m.016ywr/88-FaceId-0_align...,0.030028,1.0
2,database3/database3/m.02dfmb/105-FaceId-0_alig...,0.009984,0.0
3,database3/database3/m.01_h_s/33-FaceId-5_align...,0.255016,1.0
4,database3/database3/m.02lqkv/85-FaceId-0_align...,0.252091,1.0


In [7]:
y = df_train["FaceOcclusion"].to_numpy()
gender_np = df_train["gender"].to_numpy()
bin_idx = np.clip(np.digitize(y, bins, right=False) - 1, 0, len(w_f) - 1)
gw = np.where(gender_np == 0.0, w_f[bin_idx], w_m[bin_idx])

In [8]:
df_train["gender_weigth"] = gw


In [9]:
df_train.head()

,filename,FaceOcclusion,gender,gender_weigth
0,database3/database3/m.01w0zk1/66-FaceId-0_alig...,0.024005,1.0,0.569136
1,database3/database3/m.016ywr/88-FaceId-0_align...,0.030028,1.0,0.569136
2,database3/database3/m.02dfmb/105-FaceId-0_alig...,0.009984,0.0,4.116060
3,database3/database3/m.01_h_s/33-FaceId-5_align...,0.255016,1.0,1.102514
4,database3/database3/m.02lqkv/85-FaceId-0_align...,0.252091,1.0,1.102514


In [10]:
df_train.to_csv("../data/occlusion_datasets/train_gender_weights.csv", sep=',', index=False)

In [11]:
df_train_gender = pd.read_csv("../data/occlusion_datasets/train_gender_weights.csv", delimiter=',')
df_train_gender.head()

,filename,FaceOcclusion,gender,gender_weigth
0,database3/database3/m.01w0zk1/66-FaceId-0_alig...,0.024005,1.0,0.569136
1,database3/database3/m.016ywr/88-FaceId-0_align...,0.030028,1.0,0.569136
2,database3/database3/m.02dfmb/105-FaceId-0_alig...,0.009984,0.0,4.116060
3,database3/database3/m.01_h_s/33-FaceId-5_align...,0.255016,1.0,1.102514
4,database3/database3/m.02lqkv/85-FaceId-0_align...,0.252091,1.0,1.102514


In [16]:
import torch 
torch.tensor(bins)

tensor([0.0000, 0.0333, 0.0667, 0.1000, 0.1333, 0.1667, 0.2000, 0.2333, 0.2667,
        0.3000, 0.3333, 0.3667, 0.4000, 0.4333, 0.4667, 0.5000, 0.5333, 0.5667,
        0.6000, 0.6333, 0.6667, 0.7000, 0.7333, 0.7667, 0.8000, 0.8333, 0.8667,
        0.9000, 0.9333, 0.9667, 1.0000], dtype=torch.float64)